# ️ Poradnik: Jak łączyć tabele (Merge) w Pandas

> [!info]  Klaudia i Jakub ( patryk zakładam że pracowałeś na bazie danych  wielo tabelowej)
> Pracujemy na bazie relacyjnej (`nycflights13`). To oznacza, że zamiast jednego wielkiego pliku, w którym wszystko się powtarza, mamy jedną **Tabelę Główną (Loty)** i kilka mniejszych **Tabel Słownikowych (Wymiarów)**.
>
> Aby wyciągnąć pełne informacje (np. zamienić kod linii lotniczej na jej pełną nazwę), musimy te tabele ze sobą **złączyć** (zrobić tzw. JOIN/MERGE). W Pandas służy do tego funkcja `.merge()`.
> Po wykonaniu merge, nowe kolumny ze słownika (np. pełna nazwa linii) zostaną doklejone na samym końcu Waszej tabeli, z prawej strony
---

## 1. Dwie najważniejsze zasady łączenia
(w pliku info_do_bazy macie opisanie jak zachodzą relacje między tabelami by wiedzić ko jakich kolumnacyh joinować)
Przed napisaniem kodu musicie zdecydować o dwóch rzeczach:
1. **Po czym łączymy? (`on`)** – Szukamy kolumny, która występuje w obu tabelach i ma takie same wartości (tzw. klucz).
2. **Jak łączymy? (`how`)** – Zazwyczaj używamy **`how='left'`** (Left Join).
   * *Dlaczego Left?* Chcemy zachować wszystkie nasze loty z tabeli głównej (lewej) i tylko "dokleić" do nich informacje z tabeli słownikowej (prawej). Nawet jeśli dla jakiegoś lotu nie znajdziemy dopasowania w słowniku, lot nie zniknie z naszej bazy (pojawią się po prostu wartości `NaN`).



---

## 2. Przykłady w kodzie (Krok po Kroku)

### Scenariusz A: Kolumny klucza nazywają się tak samo
Najprostszy przypadek. Chcemy dodać pełną nazwę linii lotniczej z tabeli `airlines` do tabeli `flights`. W obu tabelach klucz nazywa się `carrier`.

```python
import pandas as pd

# how='left' -> trzymamy wszystkie wiersze z tabeli 'flights'
# on='carrier' -> kolumna łącząca
flights_z_liniami = pd.merge(flights, airlines, on='carrier', how='left')

# Można to zapisać też krócej, wywołując metodę bezpośrednio na dataframe:
# flights_z_liniami = flights.merge(airlines, on='carrier', how='left')

```

### Scenariusz B: Kolumny klucza mają RÓŻNE nazwy
Chcemy dodać dane o lotnisku docelowym z tabeli airports. W tabeli lotów klucz to dest, a w tabeli lotnisk klucz to faa. Nie możemy użyć on, musimy wskazać palcem lewą i prawą stronę.

```python
# left_on -> jak nazywa się klucz w głównej tabeli (flights)
# right_on -> jak nazywa się klucz w tabeli słownikowej (airports)
flights_z_lotniskami = pd.merge(flights, airports, left_on='dest', right_on='faa', how='left')

# Dobra praktyka: Pandas zostawi obie kolumny ('dest' i 'faa').
# Warto od razu usunąć zduplikowaną kolumnę z kluczem:
flights_z_lotniskami = flights_z_lotniskami.drop(columns=['faa'])

```

### Scenariusz C: Łączenie po WIELU kolumnach naraz
Chcemy przypiąć warunki pogodowe z tabeli weather do konkretnego startu w tabeli flights. Żeby to zrobić precyzyjnie, musimy dopasować lotnisko, rok, miesiąc, dzień i godzinę jednocześnie!

```python
# Klucze podajemy jako listę w nawiasach kwadratowych
klucze_pogody = ['origin', 'year', 'month', 'day', 'hour']

flights_z_pogoda = pd.merge(flights, weather, on=klucze_pogody, how='left')
```
### Scenariusz D: Podwójne złączenie z tą samą tabelą (Problem _x i _y)
W docelowej analizie konieczne będzie złączenie tabeli `flights` z tabelą `airports` dwukrotnie: najpierw w celu pobrania danych lotniska docelowego (`dest`), a następnie lotniska startowego (`origin`).

Gdy dołączysz tę samą tabelę słownikową po raz drugi, Pandas napotka konflikt nazw kolumn (np. kolumna `name` dla lotniska docelowego i `name` dla lotniska startowego). Domyślnie system doda do nich mało czytelne sufiksy `_x` oraz `_y`. Aby zachować porządek, należy użyć parametru `suffixes`.

```python
# 1. Pierwsze złączenie (dla lotniska docelowego)
flights_krok1 = pd.merge(flights, airports, left_on='dest', right_on='faa', how='left')
flights_krok1 = flights_krok1.drop(columns=['faa'])

# 2. Drugie złączenie (dla lotniska startowego)
# Parametr suffixes dodaje przyrostki do powtarzających się nazw kolumn.
# Lewy element krotki dotyczy kolumn już istniejących w tabeli głównej (z pierwszego złączenia),
# a prawy element dotyczy kolumn z nowo dołączanej tabeli słownikowej.
flights_pelne = pd.merge(flights_krok1, airports, left_on='origin', right_on='faa', how='left', suffixes=('_cel', '_start'))
flights_pelne = flights_pelne.drop(columns=['faa'])

# Wynik: zamiast 'name_x' i 'name_y' otrzymasz czytelne 'name_cel' oraz 'name_start'

```


## 3. Przez co merge może nie działać

### Czy typy danych (dtypes) w kluczach są takie same?
Klucz w jednej tabeli może być traktowany przez Pythona jako tekst (`object`/`string`), a w drugiej jako liczba (`int`/`float`).
Jeśli spróbujecie połączyć tekst z liczbą, Pandas wyrzuci błąd lub po prostu nie znajdzie dopasowań, zasypując wynikową tabelę brakami.

Jak sprawdzić i naprawić:
```python
# Sprawdzenie typów
print("Typ w flights:", flights['dest'].dtype)
print("Typ w airports:", airports['faa'].dtype)

# Wymuszenie tego samego typu (np. na tekst) tuż przed mergem
flights['dest'] = flights['dest'].astype(str)
airports['faa'] = airports['faa'].astype(str)
```

### Czy w tekstach nie pochowały się niewidoczne spacje?
Czasami przy wczytywaniu danych kod linii lotniczej w głównej tabeli to `"UA"`, a w tabeli pobocznej `"UA "`. Dla komputera to dwie zupełnie różne wartości. Efekt? Poprawny kod, zero błędów w konsoli, ale po złączeniu macie same `NaN`.

Jak sprawdzić i naprawić:
```python
# Wypisanie unikalnych wartości (widać wtedy dziwne spacje na końcach)
print(flights['carrier'].unique())

# Obcięcie niewidocznych białych znaków z przodu i z tyłu (bezpieczna praktyka)
flights['carrier'] = flights['carrier'].str.strip()
airlines['carrier'] = airlines['carrier'].str.strip()
```

### Czy tabela słownikowa na pewno nie ma duplikatów?
Jeśli zrobicie `left join`, a w prawej tabeli (słownikowej) ten sam klucz wystąpi dwa razy, Pandas zduplikuje wiersze w tabeli głównej. Nagle z 336 tysięcy lotów zrobi się Wam 400 tysięcy, a Wasze statystyki i końcowe wizualizacje wyjdą kompletnie zafałszowane.

Jak sprawdzić:
```python
# Zwróci True, jeśli jakikolwiek klucz w tabeli słownikowej się powtarza
czy_sa_duplikaty = airlines['carrier'].duplicated().any()
print("Czy są duplikaty w słowniku?", czy_sa_duplikaty)

# Podgląd konkretnych zduplikowanych wierszy (jeśli na górze wyszło True)
print(airlines[airlines['carrier'].duplicated(keep=False)])
# Usunięcie duplikatów ze słownika (zostawia tylko pierwsze wystąpienie)( sprawdza TYLKO i wyłącznie kolumnę (lub kolumny) podane w parametrze subset )
airlines = airlines.drop_duplicates(subset=['carrier'])
```

### Czy kolumna, po której łączycie (`on`), ma identyczną nazwę w obu tabelach?
Klasyczny błąd `KeyError`. Często wydaje się nam, że kolumna nazywa się tak samo, ale w tabeli z lotami to `dest`, a w tabeli z lotniskami to `faa`. Użycie samego parametru `on='dest'` wywali natychmiastowy błąd.

Jak sprawdzić:
```python
# Szybki podgląd list z nazwami wszystkich kolumn
print("Kolumny flights:", flights.columns.tolist())
print("Kolumny airports:", airports.columns.tolist())

# Pamiętajcie: jeśli nazwy są różne, zamiast 'on' używacie 'left_on' i 'right_on'
```